# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/codingsheep17/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
#initialising the repo
import os

if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/codingsheep17/flyrank-ml-internship.git

os.chdir("flyrank-ml-internship")
print(os.getcwd())

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 209, done.
remote: Counting objects: 100% (209/209), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 209 (delta 95), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (209/209), 2.48 MiB | 6.01 MiB/s, done.
Resolving deltas: 100% (95/95), done.
/content/flyrank-ml-internship/flyrank-ml-internship


In [17]:
#installing the dataset
!pip install duckdb huggingface_hub -q

from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("Token loaded:", "HF_TOKEN" in os.environ)

Token loaded: True


In [18]:
import duckdb

con = duckdb.connect()
con.sql(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")
print("DuckDB secret configured")

DuckDB secret configured


In [19]:
con.sql("""
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
LIMIT 5
""").show()

┌─────────────────────────┬───────────┬────────────────┬────────────────┬───────────────────────────────┬─────────────────────┬─────────────────────┬────────────────┬────────────────┐
│     client_hash_id      │ is_active │ has_gsc_access │ has_ga4_access │        access_profile         │ client_created_date │ client_updated_date │ gsc_data_start │ ga4_data_start │
│         varchar         │  boolean  │    boolean     │    boolean     │            varchar            │        date         │        date         │      date      │      date      │
├─────────────────────────┼───────────┼────────────────┼────────────────┼───────────────────────────────┼─────────────────────┼─────────────────────┼────────────────┼────────────────┤
│ client_04660893ae39614a │ true      │ true           │ true           │ gsc_and_ga4                   │ 2026-04-15          │ 2026-06-27          │ NULL           │ 2026-05-22     │
│ client_05475c07ed21a83a │ true      │ false          │ false          │ no_sea

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

. What one row means for my lane: One row = one page's daily performance (a single content item on a single day)

2. Which table(s) I'll use: fact_content_daily_performance (main daily metrics), joined with dim_content (content metadata) on content_hash_id

3. Time window: One mid-panel month, month=2026-03, used for developing and iterating — never the final month (2026-06), which is the sealed test window

4. What I'd predict or rank (label/proxy): Whether a page's performance is declining — a proxy built from comparing recent daily metrics within the month (e.g. trend direction based on impressions/clicks), similar to the trend_direction proxy used in the starter dataset

5. One thing I deliberately exclude: I exclude any FlyRank product decision flags (like health_score, priority_score, action_type) — these aren't in the released data anyway, but I confirm I will not attempt to reconstruct them as features, since doing so would let the model copy an existing decision instead of learning real signal

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = con.sql("""
SELECT
    content_hash_id,
    report_date,
    COUNT(*) as row_count
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
GROUP BY content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 10
""").df()

print(f"Duplicate (content, date) combinations found: {len(grain_check)}")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (content, date) combinations found: 0


,content_hash_id,report_date,row_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature fields (usable inputs): impressions, clicks, avg_position, ctr (from GSC), sessions, engagement_rate (from GA4) — all observable signals known at the decision moment

Label/proxy field: a decline label I'll construct myself by comparing earlier vs. later performance within the month (not a pre-built column) — following the same principle as the starter dataset's trend_direction

Context fields (for grouping/joins, not features): content_hash_id, client_hash_id, report_date

Excluded fields, with reason: any FlyRank product decision flags (health_score, priority_score, action_type) — not present in this release, and I will not attempt to reconstruct them, since doing so would let a model copy an existing decision instead of learning real signal (circular result risk)

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
counts_check = con.sql("""
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as earliest_date,
    MAX(report_date) as latest_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()

counts_check


,total_rows,earliest_date,latest_date
0,9841378,2026-03-01,2026-03-31


In [23]:
schema_check = con.sql("""
DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet' LIMIT 1
""").df()

print(schema_check.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [24]:
availability_check = con.sql("""
SELECT
    COUNT(*) as total_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) as rows_with_ga4,
    COUNT(*) FILTER (WHERE gsc_impressions IS NOT NULL) as rows_with_impressions
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_ga4,rows_with_impressions
0,9841378,413966,9841378


In [25]:
impressions_check = con.sql("""
SELECT
    COUNT(*) as total_rows,
    COUNT(*) FILTER (WHERE gsc_impressions > 0) as rows_with_real_impressions,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) as rows_with_gsc_available
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()

impressions_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_real_impressions,rows_with_gsc_available
0,9841378,3611061,3611061


Feature frame (5 features max) for March 2026

In [26]:
features_df = con.sql("""
SELECT
    content_hash_id,
    client_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
USING SAMPLE 5000 ROWS
""").df()

features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,content_0a72369e80dd0376,client_23a62021009f63c4,2026-03-08,75,0,37.786667,0,0
1,content_d0d52c7ff7217dac,client_9958f0a7ae1df715,2026-03-22,2,0,24.000000,0,0
2,content_9c1985f6c5978cce,client_e547b89c05043229,2026-03-13,12,0,6.916667,0,0
3,content_a07d6b4935cda471,client_400c21c81c8b46ef,2026-03-25,10,0,7.200000,0,0
4,content_2bf9ed10b86d6a50,client_62f4a7e64f5e0096,2026-03-08,3,0,25.666667,<NA>,<NA>


Feature availability ("knowable at the decision moment because…"):

gsc_impressions — knowable because it's a daily search-visibility count already recorded once the day's search data syncs, before any decision is made
gsc_clicks — same as impressions, a completed daily count from Search Console, not a future value
gsc_avg_position — knowable because it's calculated from completed search impressions for that day, not a projection
ga4_sessions — knowable when available, since it's a completed daily engagement count, though it's frequently missing (as shown above) for pages without GA4 tracking
ga4_engaged_sessions — same as sessions, a completed daily count, also frequently missing

The trap: deliberate leakage experiment

In [28]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Build a simple proxy label: "declining" = below-median clicks relative to impressions (crude CTR-based proxy)
df = features_df.copy()
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"].replace(0, np.nan)
df["ctr"] = df["ctr"].fillna(0)

# Use mean instead of median, since CTR is heavily zero-inflated
threshold = df["ctr"].mean()
df["label_declining"] = (df["ctr"] < threshold).astype(int)

print(df["label_declining"].value_counts())

# Honest features only (no leakage)
X_honest = df[["gsc_impressions", "gsc_avg_position"]].fillna(0)
y = df["label_declining"]

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model = LogisticRegression().fit(X_train, y_train)
honest_score = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
print(f"Honest ROC AUC (no leakage): {honest_score:.3f}")

label_declining
1    1663
0     194
Name: count, dtype: int64
Honest ROC AUC (no leakage): 0.821


In [29]:
# THE TRAP: add a label-derived feature on purpose (this is fake, done deliberately to prove the leakage lesson)
df["leaky_feature"] = df["ctr"] * 100  # directly derived from the same value used to build the label

X_leaky = df[["gsc_impressions", "gsc_avg_position", "leaky_feature"]].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
model_leaky = LogisticRegression().fit(X_train, y_train)
leaky_score = roc_auc_score(y_test, model_leaky.predict_proba(X_test)[:,1])
print(f"LEAKY ROC AUC (with leakage): {leaky_score:.3f}")

LEAKY ROC AUC (with leakage): 0.999


The leakage lesson: Adding leaky_feature (directly derived from the same CTR value used to build the label) pushed ROC AUC from 0.821 to 0.999 — an unrealistic, too-good-to-be-true jump. This is a textbook example of label leakage: the model wasn't learning a real pattern, it was just handed the answer through a disguised backdoor. The leaky feature has been removed. The honest, trustworthy number for this data is 0.821, not 0.999.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One named limitation of this slice: GA4 (engagement/session) data is only available for a small fraction of rows (4.2%, 413,966 out of 9.84M) in this month, while GSC (search) availability is much broader (~7%, 3.61M rows). This means any feature relying on engagement metrics (sessions, scroll rate) will only be usable for a small subset of pages, and models trained using those features may not generalize well to pages without GA4 tracking. This is an unbalanced panel, as described in the lane guide — different clients have different tracking coverage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


✅ 5 plain-words contract answers (Section 1) — done
✅ Fields sorted into feature/label/context/excluded (Section 2) — done
✅ Exactly 3 verification queries with visible outputs, availability checked with IS TRUE (Section 3) — done (grain, row count/dates, availability)
✅ 5-feature frame with "available when" line per feature — done
✅ Deliberate leak experiment shown and removed (0.821 → 0.999 → back to 0.821) — done
✅ One named limitation (Section 4 — GA4 sparsity) — done